# Week 52, Day 3: Backtracking

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w052d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def combos(items, k):
    out, current = [], []
    def build(start):
        if len(current) == k:
            out.append(current)            # careful...
            return
        for i in range(start, len(items)):
            current.append(items[i])
            build(i + 1)
            current.pop()
    build(0)
    return out

print(combos("abc", 2))

In [ ]:
def combination_sum(numbers, target):
    numbers = sorted(numbers)
    out, current = [], []
    def build(start, remaining):
        if remaining == 0:
            out.append(list(current))                 # 1. done: record a copy
            return
        for i in range(start, len(numbers)):
            n = numbers[i]
            if n > remaining:
                break                                  # sorted, so every later number is too big
            current.append(n)                          # choose
            build(i, remaining - n)                    # explore (i, not i + 1: n may be reused)
            current.pop()                              # unchoose
    build(0, target)
    return out

print(combination_sum([2, 3, 6, 7], 7), combination_sum([2, 3, 5], 8))

In [ ]:
def count_nodes(n):
    """How many partial permutations a full permutation search visits."""
    calls = [0]
    used = [False] * n
    def build(depth):
        calls[0] += 1
        if depth == n:
            return
        for i in range(n):
            if not used[i]:
                used[i] = True
                build(depth + 1)
                used[i] = False
    build(0)
    return calls[0]

for n in range(1, 9):
    print(n, count_nodes(n))

In [ ]:
def unique_permutations(items):
    items = sorted(items)
    out, current, used = [], [], [False] * len(items)
    def build():
        if len(current) == len(items):
            out.append("".join(current))
            return
        for i in range(len(items)):
            if used[i]:
                continue
            if i > 0 and items[i] == items[i - 1] and not used[i - 1]:
                continue                     # an equal item must be placed first
            used[i] = True
            current.append(items[i])
            build()
            current.pop()
            used[i] = False
    build()
    return out

print(unique_permutations("aab"), len(unique_permutations("aabb")))

In [ ]:
def find_subset(numbers, target):
    current = []
    def build(i, remaining):
        if remaining == 0:
            return True
        if i == len(numbers) or remaining < 0:
            return False
        current.append(numbers[i])
        if build(i + 1, remaining - numbers[i]):
            return True                         # pass success straight up: stop searching
        current.pop()
        return build(i + 1, remaining)          # or try without numbers[i]
    return list(current) if build(0, target) else None

print(find_subset([8, 6, 7, 5, 3, 10, 9], 15), find_subset([2, 4, 6], 5))

In [ ]:
def brackets(n):
    out, current = [], []
    def build(opened, closed):
        if len(current) == 2 * n:
            out.append("".join(current))
            return
        if opened < n:
            current.append("(")
            build(opened + 1, closed)
            current.pop()
        if closed < opened:
            current.append(")")
            build(opened, closed + 1)
            current.pop()
    build(0, 0)
    return out

print(brackets(3), len(brackets(6)))

> **Key idea.** Backtracking searches a tree of partial solutions depth first with one shared partial solution: if it's complete, record a copy; otherwise for each allowed choice, choose it, explore, then unchoose. The tree's size (combinations, factorials, powers) is the cost, so prune: reject choices that break a constraint as early as possible, and stop at the first solution when one is enough by passing success back up. With repeated items, sort and skip a value equal to the previous one at the same level (for permutations, unless the previous equal item is in use). Record `list(current)`, never `current` itself.

### Exercise: Combinations that add up

Write `combination_sum(numbers, target)` (distinct positive numbers, each usable any number of times; every combination as a sorted list, all of them in sorted order) and `combination_sum_once(numbers, target)`: each number in the list used at most **once** (the list may contain repeats, but each combination must appear only once).

In [ ]:
def combination_sum(numbers, target):
    ___

def combination_sum_once(numbers, target):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_combos():\n    \"combinations\"\n    assert combination_sum([2, 3, 6, 7], 7) == [[2, 2, 3], [7]] and combination_sum([2], 1) == []\n    assert combination_sum_once([10, 1, 2, 7, 6, 1, 5], 8) == [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]\n    rng = random.Random(1)\n    for _ in range(60):\n        nums = [rng.randint(1, 6) for _ in range(rng.randint(0, 7))]\n        t = rng.randint(1, 12)\n        want = sorted({tuple(sorted(c)) for r in range(len(nums) + 1) for c in itertools.combinations(nums, r) if sum(c) == t})\n        assert combination_sum_once(nums, t) == [list(c) for c in want]")

### Exercise: Arrangements without repeats

Write `unique_permutations(items)`: every distinct arrangement of a string's letters, as strings in sorted order, without generating duplicates (sort, and skip an equal letter whose earlier twin isn't in use). Then `unique_subsets(items)`: every distinct sub-collection of a list (as sorted lists, the empty one included), without duplicates, in the order a search with a `start` index finds them.

In [ ]:
def unique_permutations(items):
    ___

def unique_subsets(items):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_perms():\n    \"permutations and subsets\"\n    assert unique_permutations(\"aab\") == [\"aab\", \"aba\", \"baa\"] and unique_permutations(\"\") == [\"\"]\n    assert unique_subsets([1, 2, 2]) == [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]\n    rng = random.Random(2)\n    for _ in range(40):\n        s = \"\".join(rng.choice(\"abc\") for _ in range(rng.randint(0, 6)))\n        assert unique_permutations(s) == sorted({\"\".join(p) for p in itertools.permutations(s)})\n        xs = [rng.randint(1, 3) for _ in range(rng.randint(0, 6))]\n        want = sorted({tuple(sorted(c)) for r in range(len(xs) + 1) for c in itertools.combinations(xs, r)})\n        assert sorted(tuple(x) for x in unique_subsets(xs)) == want")

### Exercise: Balanced brackets

Write `brackets(n)`: every balanced string of n pairs of `(` and `)`, in the order the pruned search finds them (open first), never building an invalid prefix. Then `count_brackets(n)`: just the number, using a counting search without building strings, and check it's the Catalan number for small n.

In [ ]:
def brackets(n):
    ___

def count_brackets(n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_brackets():\n    \"brackets\"\n    assert brackets(3) == [\"((()))\", \"(()())\", \"(())()\", \"()(())\", \"()()()\"] and brackets(0) == [\"\"]\n    assert [count_brackets(n) for n in range(9)] == [1, 1, 2, 5, 14, 42, 132, 429, 1430]\n    for s in brackets(5):\n        depth = 0\n        for ch in s:\n            depth += 1 if ch == \"(\" else -1\n            assert depth >= 0\n        assert depth == 0")

### Exercise: Cutting into palindromes

Write `palindrome_cuts(s)`: every way to cut s into pieces that are all palindromes, each as a list of pieces, in the order a left-to-right search finds them (shorter first piece first). Precompute which substrings are palindromes in a table so each check is O(1). Then `fewest_cuts(s)`: the fewest pieces (not cuts) needed, using a search that **stops exploring** any branch already using at least as many pieces as the best found so far.

In [ ]:
def palindrome_cuts(s):
    ___

def fewest_cuts(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_palins():\n    \"palindromes\"\n    assert palindrome_cuts(\"aab\") == [[\"a\", \"a\", \"b\"], [\"aa\", \"b\"]] and palindrome_cuts(\"\") == [[]]\n    assert fewest_cuts(\"aab\") == 2 and fewest_cuts(\"racecar\") == 1 and fewest_cuts(\"abc\") == 3\n    rng = random.Random(3)\n    for _ in range(60):\n        s = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(1, 10)))\n        cuts = palindrome_cuts(s)\n        assert all(\"\".join(c) == s and all(p == p[::-1] for p in c) for c in cuts)\n        assert fewest_cuts(s) == min(len(c) for c in cuts)")

### Exercise: Colouring a map

Write `colour_map(adj, k)`: a dict giving each region (vertex) one of the colours 0 to k − 1 so that neighbours never share a colour, or `None` if impossible. Colour regions in the dict's order, trying colours from 0 upwards, and reject a colour as soon as a coloured neighbour has it. Then `fewest_colours(adj)`: the smallest k that works (the **chromatic number**).

In [ ]:
def colour_map(adj, k):
    ___

def fewest_colours(adj):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_colour():\n    \"colouring\"\n    triangle = {1: [2, 3], 2: [1, 3], 3: [1, 2]}\n    assert colour_map(triangle, 2) is None and colour_map(triangle, 3) == {1: 0, 2: 1, 3: 2}\n    square = {1: [2, 4], 2: [1, 3], 3: [2, 4], 4: [3, 1]}\n    assert fewest_colours(square) == 2 and fewest_colours({1: []}) == 1 and fewest_colours({}) == 0\n    rng = random.Random(4)\n    for _ in range(40):\n        n = rng.randint(1, 6)\n        g = {v: [] for v in range(n)}\n        for a, b in itertools.combinations(range(n), 2):\n            if rng.random() < 0.5:\n                g[a].append(b)\n                g[b].append(a)\n        k = fewest_colours(g)\n        c = colour_map(g, k)\n        assert c is not None and all(c[a] != c[b] for a in g for b in g[a])\n        assert k == 1 or colour_map(g, k - 1) is None")

### Bug hunt: The same list every time

This records `current` itself, not a copy, so every result is the same list, emptied as the search backs out: this prints `[[], [], []]`. Record a copy so it prints `[['a', 'b'], ['a', 'c'], ['b', 'c']]`.

In [ ]:
def combos(items, k):
    out, current = [], []
    def build(start):
        if len(current) == k:
            out.append(current)
            return
        for i in range(start, len(items)):
            current.append(items[i])
            build(i + 1)
            current.pop()
    build(0)
    return out

print(combos("abc", 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"copy it\"\n    expect_output(\"[['a', 'b'], ['a', 'c'], ['b', 'c']]\")")

### Bug hunt: Never stepping back

This subset search adds each item but never removes it after exploring, so later branches start with leftovers: it prints far too many, wrong subsets. Unchoose after exploring so it prints `[[], ['a'], ['a', 'b'], ['b']]`.

In [ ]:
def subsets(items):
    out, current = [], []
    def build(start):
        out.append(list(current))
        for i in range(start, len(items)):
            current.append(items[i])
            build(i + 1)
    build(0)
    return out

print(subsets("ab"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"unchoose\"\n    expect_output(\"[[], ['a'], ['a', 'b'], ['b']]\")")

### Bug hunt: Duplicates from duplicates

The list has two 2s, so this finds the subset `[2]` twice and `[1, 2]` twice: it prints `6`. Skip a value equal to the previous one at the same level (`i > start and items[i] == items[i - 1]`), so it prints `4`.

In [ ]:
items = sorted([2, 1, 2])
out, current = [], []

def build(start):
    if len(current) in (1, 2):
        out.append(list(current))
    for i in range(start, len(items)):
        current.append(items[i])
        build(i + 1)
        current.pop()

build(0)
print(len(out))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"skip equal siblings\"\n    expect_output(\"4\")")

### Puzzle: The recipe

Arrange the lines into a backtracking search for combinations of k items. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the number of leaves (complete permutations) when ordering 5 distinct items
q = ___   # the number of distinct arrangements of "aabb"
r = ___   # combination_sum([2, 3], 6), each number reusable, as a list of lists
s = ___   # the number of balanced strings of 4 pairs of brackets
t = ___   # the fewest colours for a cycle of 5 regions in a ring

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 120 and q == 6 and r == [[2, 2, 2], [3, 3]], \"p, q and r\"\n    assert s == 14 and t == 3, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: equal shares

Can a list of positive whole numbers be split into k groups with **equal sums**? (Sharing out jobs fairly among k people, packing equal loads.) Write `equal_shares(numbers, k)`: a list of k lists with equal sums using every number exactly once, or `None`. Backtrack by placing each number, **largest first**, into one of the k groups, with three prunings:

- give up at once if the total isn't divisible by k, or the largest number exceeds the target share;
- never put a number into a group where it would exceed the share;
- if a number doesn't fit into an **empty** group, it won't fit into any other empty group either (they're identical), so try only the first empty group; more generally, skip a group whose current sum equals one already tried for this number.

The checks include 16 numbers into 4 groups, which the unpruned search would take far too long on.

In [ ]:
def equal_shares(numbers, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time, itertools\ndef check(numbers, k, groups):\n    return groups is not None and len(groups) == k and sorted(x for g in groups for x in g) == sorted(numbers) and len({sum(g) for g in groups}) == 1\ndef test_shares():\n    \"shares\"\n    assert check([4, 3, 2, 3, 5, 2, 1], 4, equal_shares([4, 3, 2, 3, 5, 2, 1], 4))\n    assert equal_shares([1, 2, 3, 4], 3) is None and equal_shares([5, 1, 1], 2) is None\n    assert check([7], 1, equal_shares([7], 1))\n    rng = random.Random(5)\n    for _ in range(60):\n        nums = [rng.randint(1, 9) for _ in range(rng.randint(1, 8))]\n        k = rng.randint(1, 3)\n        got = equal_shares(nums, k)\n        possible = False\n        if sum(nums) % k == 0:\n            for labels in itertools.product(range(k), repeat=len(nums)):\n                sums = [0] * k\n                for x, l in zip(nums, labels):\n                    sums[l] += x\n                if len(set(sums)) == 1:\n                    possible = True\n                    break\n        assert (got is not None) == possible and (got is None or check(nums, k, got))\ndef test_hard():\n    \"pruning matters\"\n    nums = [10, 10, 10, 7, 7, 7, 7, 7, 7, 6, 6, 6, 2, 1, 3, 4]\n    t0 = time.perf_counter()\n    groups = equal_shares(nums, 4)\n    assert check(nums, 4, groups) and time.perf_counter() - t0 < 3\n    t0 = time.perf_counter()\n    assert equal_shares([9] * 15 + [2], 4) is None and time.perf_counter() - t0 < 3")

## Warm-up

Three empty lists. The search worked, but every result is the **same** list object, `current`, which was emptied as the search backed out. Appending `list(current)` (a copy) fixes it. It's the most common backtracking bug.

## The recipe

Every backtracking search follows the same shape, a recursive function over a **partial solution**:

1. **Done?** If the partial solution is complete, record it (a **copy**) and return.
2. For each **choice** available from here:
   - **Skip** it if it breaks a constraint (this is the pruning that makes backtracking fast);
   - **Choose**: add it to the partial solution;
   - **Explore**: recurse;
   - **Unchoose**: remove it again, restoring the partial solution exactly as it was.

The unchoose step is what makes it **back**tracking: one shared partial solution is reused for the whole search, instead of building a new one at every step.

Here's the recipe for "every combination of numbers that adds up to a target, each number usable any number of times". The `start` index stops the same combination being found in different orders (2 + 3 and 3 + 2), and sorting lets the loop stop as soon as a number is too big.

## The search tree

Picture the search as a **tree**: the root is the empty partial solution, each child adds one choice, and the leaves are complete solutions or dead ends. Backtracking is a depth-first walk of that tree (Week 50 Day 3) that never builds the tree in memory, only the current path.

The tree's size is the cost. Choosing k of n items gives C(n, k) leaves; ordering n items gives n! leaves; filling n slots with b choices each gives bⁿ. These numbers grow very fast (12! is about 479 million), which is why **pruning**, cutting off whole subtrees early, decides whether a backtracking search finishes in a second or never.

## Repeated items

With repeated items, the search finds the same answer several times: permutations of "aab" include "aab" twice (with the two a's swapped). Generating everything and removing duplicates with a set works, but wastes time and memory. Better: **sort**, and at each level of the tree, never try the **same value** twice:

- For combinations and subsets (with a `start` index): skip `items[i]` when `i > start` and `items[i] == items[i - 1]`.
- For permutations (with a `used` list): skip `items[i]` when it equals `items[i - 1]` and `items[i - 1]` is **not** currently used; this forces equal items to be placed in their sorted order, so each distinct arrangement appears once.

## Finding one, counting all

Three different questions need slightly different searches:

- **All solutions**: record every complete one (as above).
- **Count solutions**: add 1 instead of recording; no copying needed.
- **Any one solution**: return `True` (or the solution) as soon as one is found, and have every level pass it straight back up, so the search stops at once instead of exploring the rest of the tree.

A common way to write "find one" is for the function to return whether it succeeded:

Notice that on success the path is left in place (no pop), so `current` holds the answer.

## Constraints as you go

The strongest pruning checks each constraint at the **earliest** moment it can fail, rather than only on complete solutions. Generating balanced bracket strings of n pairs, for example: you could generate all 2²ⁿ strings of brackets and test each, or only add `(` while fewer than n have been used and only add `)` while it wouldn't close more than are open. The second never builds an invalid prefix at all, so every leaf is a valid answer.

The count, 132 for n = 6, is a Catalan number (Week 46 Day 4) again.

## Practice

## Quiz

## Daily challenge

## Before you go

Write out by hand the search tree for the subsets of {1, 2, 3}, marking where each subset is recorded. Then mark which branches a target sum of 3 would prune. Tomorrow: N-queens and sudoku, two classic puzzles where clever bookkeeping makes backtracking fast.